In [ ]:
import os
import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
from google.oauth2 import service_account
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build as build_upload_service
SERVICE_ACCOUNT_FILE = 'feedback-automation.json'
SCOPES = ['https://www.googleapis.com/auth/drive']

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES
)

drive_service = build('drive', 'v3', credentials=credentials)

print("Google Drive was contacted✅")

Google Drive was contacted✅


In [2]:

FILE_NAME = input("Enter the file name\n: ") + ".xlsx"
FOLDER_ID = '1KNf9NFxoAaHjNFZNyZMz6u9uzbLqsP2O'

query = f"name = '{FILE_NAME}' and '{FOLDER_ID}' in parents"
results = drive_service.files().list(q=query, fields="files(id, name)").execute()
items = results.get('files', [])

if not items:
    print("No file found.")
else:
    FEEDBACK_FILE_ID = items[0]['id']
    print(f"file ID is found: {FEEDBACK_FILE_ID}")

file ID is found: 1k8goMrzTPxZaRNnb3pjb1p1wYIXvziuk


In [3]:
import io
request = drive_service.files().get_media(fileId=FEEDBACK_FILE_ID)
file_data = io.BytesIO()
downloader = MediaIoBaseDownload(file_data, request)

done = False
while not done:
    status, done = downloader.next_chunk()

file_data.seek(0)

df = pd.read_excel(file_data)
print("file Loaded is Done✅")
df.head()

file Loaded is Done✅


,Customer_ID,Feedback
0,CUST_001,"The dress looks exactly like the pictures, and..."
1,CUST_002,"I really liked the design, but the delivery to..."
2,CUST_003,"The size was smaller than I expected, so I rec..."
3,CUST_004,Great quality and fast delivery. I will defini...
4,CUST_005,"The fabric is comfortable, but the color is sl..."


In [4]:
system_prompt= """ You are a Professional Customer Service Analyst, Your task for each of them is to analyze the customer feedback and return the result in JSON format only, without any text before or after it.
if the feedback is positive, return the result as:
this is an example of the JSON format for positive feedback:
{
  "summary": "The customer is very satisfied with the product and service.",
  "sentiment": "positive",
  "product": "dress",
  "positive_factors": [
    "fabric quality",
    "product matches photos",
    "fast delivery"
  ],
  "customer_satisfaction": 5
}

 if the feedback is negative, return the result as:
this is an example of the JSON format for negative feedback:
{
  "summary": "The customer is dissatisfied with the product and service.",
  "sentiment": "negative",
  "product": "dress",
  "issues": [
    "poor quality",
    "wrong size",
    "late delivery"
  ],
  "severity": "high",
  "customer_satisfaction": "low",
  "recommended_actions": [
    "Review size chart accuracy",
    "Improve quality control",
    "Investigate delivery delays"
  ],
  "priority": "high"
}
"""



In [5]:
user_prompt = f"""Analyze all customer feedback and write an analysis for each customer, similar to the form in System Prompt., and classify it
, and return the Each one of them result in JSON format only,
without any text before or after it. this is the feedback: {df}"""

In [6]:
import ollama

response = ollama.chat(
    model = "phi3:latest", messages = [
        {"role": "system", "content": f" {system_prompt}"},
        {"role": "user", "content": f" {user_prompt}"}
    ]
)
FIRST_AI_RESULT = response["message"]["content"]
print(FIRST_AI_RESULT)

```json
{
  "Customer_ID": "CUST_001",
  "summary": "The customer is very satisfied with the product and service.",
  "sentiment": "positive",
  "product": "dress",
  "positive_factors": [
    "fabric quality",
    "product matches photos",
    "fast delivery"
  ],
  "customer_satisfaction": 5
}
```

```json
{
  "Customer_ID": "CUST_002",
  "summary": "The customer is happy with the design but not with the delivery time.",
  "sentiment": "neutral",
  "product": "dress",
  "issues": [
    "fast delivery"
  ],
  "customer_satisfaction": "medium",
  "recommended_actions": [
    "Investigate delivery delays"
  ],
  "priority": "medium"
}
```

```json
{
  "Customer_ID": "CUST_003",
  "summary": "The customer is dissatisfied due to the incorrect size of the product.",
  "sentiment": "negative",
  "product": "dress",
  "issues": [
    "wrong size"
  ],
  "severity": "high",
  "customer_satisfaction": "low",
  "recommended_actions": [
    "Review size chart accuracy",
    "Improve quality cont

In [23]:
system_prompt2= """You are a business analyst for an online clothing store.

You will receive the output of a previous AI analysis in JSON format.

Your task is to analyze this information and provide each client with actionable business insights that can help the company improve its products and customer experience.
Based only on the provided data, identify:

1. The main problem or strength.
2. The possible business impact.
3. A practical recommendation for the company.
4. The priority level: High, Medium, or Low.

"""

In [24]:
user_prompt2 = f"""As a business analyst, I need you to analyze each client's feedback and generate the following in a JSON file format.:

{{
  "main_problem_or_strength": "",
  "possible_business_impact": "",
  "practical_recommendation": "",
  "priority_level": ""
}}
I want you to apply this formula to each of their clients.
results only in JSON format, without any text before or after it.
Previous AI analysis:
{FIRST_AI_RESULT}
"""

In [25]:
response= ollama.chat (
    model = "phi3:latest", messages = [
        {"role": "system", "content": f" {system_prompt2}"},
        {"role": "user", "content": f" {user_prompt2}"}
    ]
)
SECOND_AI_RESULT = response["message"]["content"]
print(SECOND_AI_RESULT) 

```json
[
  {
    "Customer_ID": "CUST_001",
    "main_problem_or_strength": "The customer is very satisfied with the product and service.",
    "possible_business_impact": "Maintaining high customer satisfaction can lead to increased customer loyalty and repeat purchases.",
    "practical_recommendation": "Continue the excellent service and product quality that led to high customer satisfaction.",
    "priority_level": "Low"
  },
  {
    "Customer_ID": "CUST_002",
    "main_problem_or_strength": "The customer is happy with the design but not with the delivery time.",
    "possible_business_impact": "If delivery time continues to be a problem, it could lead to negative customer reviews and decreased sales.",
    "practical_recommendation": "Address and investigate delivery delays to improve customer satisfaction.",
    "priority_level": "Medium"
  },
  {
    "Customer_ID": "CUST_003",
    "main_problem_or_strength": "The customer is dissatisfied due to the incorrect size of the product

In [26]:
import json

analysis_result = []
cleaned_result = SECOND_AI_RESULT.strip()

if "```" in cleaned_result:
    json_blocks = cleaned_result.split("```")

    for block in json_blocks:
        block = block.strip()

        if block.lower().startswith("json"):
            block = block[4:].strip()

        if not block:
            continue

        try:
            parsed_result = json.loads(block)

            if isinstance(parsed_result, list):
                analysis_result.extend(parsed_result)
            else:
                analysis_result.append(parsed_result)

        except json.JSONDecodeError:
            continue
else:
    parsed_result = json.loads(cleaned_result)

    if isinstance(parsed_result, list):
        analysis_result.extend(parsed_result)
    else:
        analysis_result.append(parsed_result)

print(f"Parsed {len(analysis_result)} JSON results successfully.")

Parsed 10 JSON results successfully.


In [27]:
results_df = pd.DataFrame(analysis_result)
Resulte_file_neame = input("Enter the name of the result file: ") + ".xlsx"
results_df.to_excel( Resulte_file_neame, index=False)

print(" saved the result in Excel done ✅")

 saved the result in Excel done ✅


In [28]:
OAUTH_SCOPES = ['https://www.googleapis.com/auth/drive.file']

flow = InstalledAppFlow.from_client_secrets_file(
    'oauth_credentials.json', OAUTH_SCOPES
)

oauth_creds = flow.run_local_server(port=0)

upload_service = build_upload_service('drive', 'v3', credentials=oauth_creds)

print("Login successful✅")

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=988232927777-s301an17r3pu8g2qekt673slpg9bkfe5.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A56128%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.file&state=kWXeuspCOUeszzeJYJyKSJRWroZQou&code_challenge=D2VzxaR03Lkvs1dnkq9_1m8HcJWWp0ztzJu5yx5yNFw&code_challenge_method=S256&access_type=offline
Login successful✅


In [29]:
file_metadata = {
    'name': Resulte_file_neame,
    'parents': [FOLDER_ID]
}

media = MediaFileUpload(
    Resulte_file_neame,
    mimetype='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
)

In [30]:
uploaded_file = upload_service.files().create(
    body=file_metadata,
    media_body=media,
    fields='id'
).execute()

print(f"The file has been uploaded to Drive✅: {uploaded_file.get('id')}")

The file has been uploaded to Drive✅: 1KZjKKOX44ey1fU2hlW_Q4G_y2YV-tN-_
